# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * (Extra credit) Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [2]:
class Matrix:
    def __init__(self, n, m=None):
        if isinstance(n, list):
            self.data = n
            self.n = len(n)
            self.m = len(n[0]) if n else 0
            if any(len(row) != self.m for row in n):
                raise ValueError("All rows must have the same number of columns.")
        elif isinstance(n, int) and isinstance(m, int):
            self.n, self.m = n, m
            self.data = [[0] * m for _ in range(n)]
        else:
            raise TypeError("Invalid initialization parameters.")

    def __getitem__(self, idx):
        if isinstance(idx, tuple):
            if isinstance(idx[0], slice) or isinstance(idx[1], slice):
                rows = self.data[idx[0]] if isinstance(idx[0], slice) else [self.data[idx[0]]]
                result = [row[idx[1]] if isinstance(idx[1], slice) else [row[idx[1]]] for row in rows]
                return Matrix(result)
            return self.data[idx[0]][idx[1]]
        return self.data[idx]

    def __setitem__(self, idx, value):
        if isinstance(idx, tuple):
            self.data[idx[0]][idx[1]] = value
        else:
            self.data[idx] = value

    def shape(self):
        return self.n, self.m

    def transpose(self):
        return Matrix([[self.data[j][i] for j in range(self.n)] for i in range(self.m)])

    def row(self, n):
        return Matrix([self.data[n]])

    def column(self, m):
        return Matrix([[self.data[i][m]] for i in range(self.n)])

    def to_list(self):
        return self.data

    def block(self, n_0, n_1, m_0, m_1):
        return Matrix([row[m_0:m_1] for row in self.data[n_0:n_1]])

    def scalarmul(self, c):
        return Matrix([[c * self.data[i][j] for j in range(self.m)] for i in range(self.n)])

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix dimensions must match.")
        return Matrix([[self.data[i][j] + N[i, j] for j in range(self.m)] for i in range(self.n)])

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix dimensions must match.")
        return Matrix([[self.data[i][j] - N[i, j] for j in range(self.m)] for i in range(self.n)])

    def mat_mult(self, N):
        if self.m != N.n:
            raise ValueError("Incompatible matrix dimensions for multiplication.")
        return Matrix([[sum(self.data[i][k] * N[k, j] for k in range(self.m)) for j in range(N.m)] for i in range(self.n)])

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix dimensions must match.")
        return Matrix([[self.data[i][j] * N[i, j] for j in range(self.m)] for i in range(self.n)])

    def equals(self, N):
        return self.data == N.data

    def __eq__(self, N):
        return self.equals(N)

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, Matrix):
            return self.mat_mult(other)
        raise TypeError("Unsupported operation.")

    def __rmul__(self, other):
        return self * other

    def __add__(self, N):
        return self.add(N)

    def __sub__(self, N):
        return self.sub(N)

    def __repr__(self):
        return "\n".join(str(row) for row in self.data)

# Special Matrix Functions
def constant(n, m, c):
    return Matrix([[c] * m for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0)

def ones(n, m):
    return constant(n, m, 1)

def eye(n):
    return Matrix([[1 if i == j else 0 for j in range(n)] for i in range(n)])

# Demonstration
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])
print("Matrix A:")
print(A)
print("\nMatrix B:")
print(B)
print("\nA + B:")
print(A + B)
print("\nA - B:")
print(A - B)
print("\nA * B:")
print(A * B)
print("\nA * 2:")
print(A * 2)
print("\nTranspose of A:")
print(A.transpose())

Matrix A:
[1, 2]
[3, 4]

Matrix B:
[5, 6]
[7, 8]

A + B:
[6, 8]
[10, 12]

A - B:
[-4, -4]
[-4, -4]

A * B:
[19, 22]
[43, 50]

A * 2:
[2, 4]
[6, 8]

Transpose of A:
[1, 3]
[2, 4]
